In [ ]:
# === GOOGLE COLAB SETUP ===
import os
if 'COLAB_GPU' in os.environ:
    print("Running on Google Colab. Setting up environment...")
    !git clone https://github.com/taybro-o/btc_ai.git
    %cd btc_ai
    !pip install -q kagglehub scikit-learn matplotlib pandas numpy tensorflow
    print("Environment ready!")
else:
    print("Running locally. Skipping Colab setup.")

# BTC AI Master Notebook (PatchTST Optimization)
This notebook handles data updates (last 6 months only), feature engineering, and training a PatchTST model for 5-minute price forecasting.

## 1. Data Acquisition (Filtered to Last 6 Months)
Automatically removes data older than 6 months to optimize CPU training time.

In [11]:
import kagglehub
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers
from datetime import datetime, timedelta

DATASET_SLUG = "mczielinski/bitcoin-historical-data"
DATA_DIR = "data"
MASTER_FILE = os.path.join(DATA_DIR, "btcusdt_analysis_data.csv")

def update_and_prune_data():
    print("Updating data from Kaggle...")
    path = kagglehub.dataset_download(DATASET_SLUG)
    csv_file = None
    for root, dirs, filenames in os.walk(path):
        for filename in filenames:
            if filename.endswith(".csv"): csv_file = os.path.join(root, filename); break
    
    new_df = pd.read_csv(csv_file)
    if os.path.exists(MASTER_FILE):
        master_df = pd.read_csv(MASTER_FILE)
        df = pd.concat([master_df, new_df]).drop_duplicates(subset=['Timestamp'], keep='last')
    else:
        df = new_df
    
    # Filtering: Keep only last 6 months (~180 days)
    df['dt'] = pd.to_datetime(df['Timestamp'], unit='s')
    cutoff = df['dt'].max() - timedelta(days=180)
    df = df[df['dt'] >= cutoff].drop(columns=['dt'])
    
    if not os.path.exists(DATA_DIR): os.makedirs(DATA_DIR)
    df.to_csv(MASTER_FILE, index=False)
    print(f"Data pruned and saved. Rows: {len(df)}. Range: {cutoff} to {datetime.now()}")
    return df

df = update_and_prune_data()
df['Datetime'] = pd.to_datetime(df['Timestamp'], unit='s')
df.set_index('Datetime', inplace=True)
df.sort_index(inplace=True)

Updating data from Kaggle...
Data pruned and saved. Rows: 259201. Range: 2025-10-01 00:04:00 to 2026-03-30 14:39:47.079363


## 2. Feature Engineering
Adding technical indicators using vectorized operations.

In [12]:
def apply_indicators(df):
    # Log Returns
    df['log_return'] = np.log(df['Close'] / df['Close'].shift(1))
    
    # EMA, RSI, MACD, ADX
    df['EMA_50'] = df['Close'].ewm(span=50).mean()
    delta = df['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
    df['RSI'] = 100 - (100 / (1 + (gain / loss)))
    df['MACD'] = df['Close'].ewm(span=12).mean() - df['Close'].ewm(span=26).mean()
    
    # ADX
    tr = pd.concat([df['High']-df['Low'], abs(df['High']-df['Close'].shift(1)), abs(df['Low']-df['Close'].shift(1))], axis=1).max(axis=1)
    atr = tr.rolling(14).mean()
    plus_di = 100 * ((df['High'].diff()).clip(lower=0).rolling(14).mean() / atr)
    minus_di = 100 * ((df['Low'].diff() * -1).clip(lower=0).rolling(14).mean() / atr)
    df['ADX'] = 100 * (abs(plus_di - minus_di) / (plus_di + minus_di + 1e-9)).rolling(14).mean()
    
    df.dropna(inplace=True)
    return df

df = apply_indicators(df)
print(f"Features created. Shape: {df.shape}")

Features created. Shape: (259131, 11)


## 3. Data Preparation for PatchTST
Scaling data and creating overlapping sequences.

In [13]:
from sklearn.preprocessing import StandardScaler

LOOKBACK = 96  # 96 minutes input
FORECAST = 5   # 5 minutes prediction

features = ['Open', 'High', 'Low', 'Close', 'Volume', 'EMA_50', 'RSI', 'MACD', 'ADX', 'log_return']
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df[features])

def create_sequences(data, lookback, forecast):
    X, y = [], []
    for i in range(len(data) - lookback - forecast + 1):
        X.append(data[i : i + lookback])
        # Predict the next 5 LOG RETURNS (last column)
        y.append(data[i + lookback : i + lookback + forecast, -1]) 
    return np.array(X), np.array(y)

X, y = create_sequences(scaled_data, LOOKBACK, FORECAST)

# Split: 90% Train, 10% Val
split = int(len(X) * 0.9)
X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]
print(f"X_train shape: {X_train.shape}")

X_train shape: (233127, 96, 10)


## 4. PatchTST Model Architecture (TensorFlow)
A custom Keras implementation of the PatchTST model.

In [14]:
class PatchTST(tf.keras.Model):
    def __init__(self, num_patches, patch_len, model_dim, num_heads, num_layers, forecast_len):
        super().__init__()
        self.patch_len = patch_len
        self.num_patches = num_patches
        self.model_dim = model_dim
        
        # 1. Patching & Linear Projection
        self.linear_proj = layers.Dense(model_dim)
        self.dropout = layers.Dropout(0.1)
        
        # 2. Transformer Blocks
        self.transformer_blocks = [
            layers.MultiHeadAttention(num_heads=num_heads, key_dim=model_dim//num_heads)
            for _ in range(num_layers)
        ]
        self.layer_norms = [layers.LayerNormalization() for _ in range(num_layers * 2)]
        
        # 3. Flatten and Linear Head
        self.flatten = layers.Flatten()
        self.head = layers.Dense(forecast_len)

    def build(self, input_shape):
        # Positional Encoding created during build
        self.pos_encoding = self.add_weight(
            name="pos_enc", 
            shape=(1, self.num_patches, self.model_dim), 
            initializer="random_normal", 
            trainable=True
        )
        super().build(input_shape)

    def call(self, x):
        # x shape: (batch, lookback, features)
        batch_size = tf.shape(x)[0]
        feat_dim = tf.shape(x)[-1]
        
        # Patching: Reshape to (batch, num_patches, patch_len * features)
        x = tf.reshape(x, (batch_size, self.num_patches, self.patch_len * feat_dim))
        
        # Project patches to model dimension
        x = self.linear_proj(x)
        x = x + self.pos_encoding
        
        # Pass through Transformer
        for i, block in enumerate(self.transformer_blocks):
            attn_out = block(x, x)
            x = self.layer_norms[i*2](x + attn_out)
            x = self.dropout(x)
            x = self.layer_norms[i*2 + 1](x)
            
        x = self.flatten(x)
        return self.head(x)

# Hyperparameters
PATCH_LEN = 16
NUM_PATCHES = LOOKBACK // PATCH_LEN
MODEL_DIM = 64
NUM_HEADS = 4
NUM_LAYERS = 2

def directional_accuracy(y_true, y_pred):
    true_sign = tf.sign(y_true)
    pred_sign = tf.sign(y_pred)
    return tf.reduce_mean(tf.cast(tf.equal(true_sign, pred_sign), tf.float32))

model = PatchTST(NUM_PATCHES, PATCH_LEN, MODEL_DIM, NUM_HEADS, NUM_LAYERS, FORECAST)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4), 
    loss='mse',
    metrics=['mae', directional_accuracy]
)

# Build by calling on a sample batch instead of model.build()
sample_x = X_train[:1]
_ = model(sample_x)
model.summary()

Model: "patch_tst_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_5 (Dense)                 │ (1, 6, 64)             │        10,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ multi_head_attention_4          │ (1, 6, 64)             │        16,640 │
│ (MultiHeadAttention)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ multi_head_attention_5          │ (1, 6, 64)             │        16,640 │
│ (MultiHeadAttention)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer_normalization_8           │ (1, 6, 64)             │           128 │
│ (LayerNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer_normalization_9           │ (1, 6, 64)             │           128 │
│ (LayerNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer_normalization_10          │ (1, 6, 64)             │           128 │
│ (LayerNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer_normalization_11          │ (1, 6, 64)             │           128 │
│ (LayerNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (1, 384)               │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (1, 5)                 │         1,925 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 46,405 (181.27 KB)

 Trainable params: 46,405 (181.27 KB)

 Non-trainable params: 0 (0.00 B)

## 5. Training
Running the training process. Estimated time: ~20-30 mins per epoch on CPU.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)
]

history = model.fit(
    X_train, y_train, 
    validation_data=(X_val, y_val), 
    epochs=30, 
    batch_size=256, 
    callbacks=callbacks,
    verbose=1
)

# Plot Loss
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.legend()
plt.title('Training and Validation Loss')
plt.show()

## 6. Visualization, Forecasting & Saving
Evaluating performance in USD and saving the trained model weights.

In [ ]:
# 1. Predict on validation set
predictions = model.predict(X_val)

# 2. Inverse Scale Log Returns (last column in our feature set)
return_mean = scaler.mean_[-1]
return_std = scaler.scale_[-1]

def denormalize_returns(data):
    return (data * return_std) + return_mean

y_val_returns = denormalize_returns(y_val)
predictions_returns = denormalize_returns(predictions)

# 3. Convert Log Returns back to Prices for visualization
# To plot prices, we use: Price_t+1 = Price_t * exp(log_return_t+1)
val_start_idx = LOOKBACK + split
val_timestamps = df.index[val_start_idx : val_start_idx + len(y_val)]

# Actual prices for the validation period
y_val_actual_prices = df['Close'].iloc[val_start_idx : val_start_idx + len(y_val)].values

# For predicted prices (1-step ahead for simple visualization)
last_close_prices = df['Close'].iloc[val_start_idx - 1 : val_start_idx + len(y_val) - 1].values
predicted_prices_1step = last_close_prices * np.exp(predictions_returns[:, 0])

plot_timestamps = val_timestamps[-100:]

plt.figure(figsize=(14, 7))
plt.plot(plot_timestamps, y_val_actual_prices[-100:], label='Actual Close', color='blue', linewidth=1.5)
plt.plot(plot_timestamps, predicted_prices_1step[-100:], label='Predicted Close (1m ahead)', color='red', linestyle='--', linewidth=1.5)

plt.title('BTC Price Forecast: Actual vs Predicted (Last 100 Minutes)')
plt.ylabel('Price (USD)')
plt.xlabel('Time')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# 4. Save the model weights
model.save_weights('btc_patchtst_weights.weights.h5')
print("Model weights saved to btc_patchtst_weights.weights.h5")